# 16 · 改进 agent：先看错题，再决定做什么

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → [16 看错题 ✅] → 17 找相似病人 → 18 疾病介绍（RAG） → 19 基因工具 → 20 大考
```

> **这一章的代码在合并请求 #18 里。** 调模型的只有一次 20 道题的试跑（约 ¥1），记在 MLflow 里；200 道题的正式考试留到第 20 章，等新工具都做完一起考。下面每一格都只读记录、在本机计算，**运行这本笔记不花钱**。

## 背景：下一步该改什么

第 15 章的 agent，第 1 名对了 39%，固定流程 35.5%。逐题比 p = 0.19，还不能说它更好。

接下来改哪里？凭感觉改，常常改错地方。公司里的做法是**先做错题分析**：把答错的题一道道归类，看哪一类最多，再针对它动手。这一章就做这件事，并据此定下后面几章要做的工具。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 错题分析：答错的 122 道错在哪 | MLflow 追踪记录 | 不凭感觉改 |
| 2 | 新开局：一开始就把固定流程的证据摆出来 | 改第一条消息和提示词 | 列表里有、却没细查的题 |
| 3 | 列不满 10 个就补满 | 按工具的顺序补 | 前 10 名的位置空着 |
| 4 | 小考：20 道题试跑 | `eval run --split ci-20` | 改完能不能跑通 |
| 5 | 定下工具清单和考法 | 消融实验 | 工具不堆，每个都要对得上一类错题 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 错题分析

### 在干什么
把 agent 在 dev-200 上第 1 名答错的题，按"错在哪一步"分成四类。

### 背景
agent 答一道题分两步：先**找到**候选（工具给出的，或它自己想到的），再从里面**选**出第一名。错在"没找到"和错在"没选对"，要补的东西完全不同。

追踪记录里存着每道题调过的每个工具和它的结果。所以不用重跑、不花钱，就能逐题查：正确答案在工具的结果里出现过没有？agent 细查过它没有？

### 用了什么技术，为什么
只读 MLflow 里第 15 章那次考试的追踪记录，再对照资料库：

| 类别 | 怎么判断 |
|---|---|
| ① 资料库里没有这个病 | 遮住这道题自己的论文以后，HPO 里这个病一条症状记录都不剩 |
| ② 资料库里有，排得太靠后 | 有记录，但没出现在任何工具的结果里 |
| ③ 看到了、细查了，没排第一 | 出现过，agent 也对它调过 `check_match` 或 `lookup_disease` |
| ④ 列表里有，没细查 | 出现过，但 agent 没细查它 |

另外数一下：每道错题的那个病，Phenopacket Store 里**别的论文**有没有报过得同一种病的病人（第 17 章要用）。

### 公司里什么情况怎么做
- **改之前先看错题**：这叫 error analysis，是做大模型应用最常被强调的一步。按类别数一数，先修最多的那类
- **错题分析写成代码**：从记录里算，换一版系统再算一次，就知道哪类变少了

### 执行
第一次运行要读一万多个病例和 HPO 资料库，大约半分钟。

In [ ]:
import collections, json
from raredx.config import settings
from raredx.data import cases as case_store, sources
from raredx.evals.harness import Harness
from raredx.knowledge import hpo as hpo_knowledge

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
cases = case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))
by_id = {c.id: c for c in cases}
papers = collections.defaultdict(set)        # 每个病：报过它的论文
for c in cases:
    papers[c.disease_id].add(c.pmid)

def latest(name, split="dev-200", **params):
    for r in harness.runs(split):
        if r["name"] == name and not r["code"].endswith("-dirty"):
            p = harness.client.get_run(r["run_id"]).data.params
            if all(p.get(k) == v for k, v in params.items()):
                return r

def why_wrong(trace):
    a = {x.name: x for x in trace.info.assessments}
    if a["disease_top1"].value:
        return None
    truth = a["disease_id"].value
    case = by_id[trace.data.spans[0].inputs["question"]["case_id"]]
    tools = [s for s in trace.data.spans if s.span_type == "TOOL"]
    if any(truth in json.dumps(s.outputs, default=str) for s in tools):
        examined = any(s.name in ("check_match", "lookup_disease") and truth in json.dumps(s.inputs, default=str)
                       for s in tools)
        kind = "③ 看到了、细查了，没排第一" if examined else "④ 列表里有，没细查"
    else:
        known = truth in hpo.disease_ids and any(not f.absent for f in hpo.features(truth, frozenset({case.pmid})))
        kind = "② 资料库里有，排得太靠后" if known else "① 资料库里没有这个病"
    return kind, bool(papers[truth] - {case.pmid})

agent_run = latest("agent", prompt_version="agent-v3")
counts, others = collections.Counter(), collections.Counter()
for trace in harness.traces(agent_run["run_id"]):
    result = why_wrong(trace)
    if result:
        counts[result[0]] += 1
        others[result[0]] += result[1]
print(f"第 1 名答错：{sum(counts.values())} 道（共 200 道）\n")
for kind in sorted(counts):
    print(f"{kind:<16} {counts[kind]:>3} 道   其中别的论文报过同病病人：{others[kind]:>3}")

### 结果怎么看

| 错在哪 | 题数 | 别的论文报过同病病人 |
|---|---|---|
| ① 资料库里没有这个病 | 38 | 2 |
| ② 资料库里有，排得太靠后 | 39 | 33 |
| ③ 看到了、细查了，没排第一 | 30 | 22 |
| ④ 列表里有，没细查 | 15 | 13 |

- **① 和 ② 加起来 77 道，占错题的六成多。问题主要在"找"，不在"选"**
- ① 的 38 道：这个病只在这道题自己的论文里描述过，遮住以后，靠症状的工具都找不到它。真实看病时论文早已发表，所以 ① 模拟的是"世界上第一个这样的病人"，医院里这种情况靠基因检测来破
- ② 的 39 道里有 33 道，别的论文报过得同一种病的病人：可以去"找相似的病人"（第 17 章）
- ④ 的 15 道：答案就在列表里，agent 却没细查。第 2 步针对它

### 面试怎么说
> "Before changing the agent I classified its 122 top-1 misses from the traces: 77 were retrieval failures — the right disease never appeared in any tool output — and 45 were judgment failures. That decided what to build next: retrieval first."

---

## 第 2 步 · 新开局：先把固定流程的证据摆出来

### 在干什么
agent 的第一条消息里，直接附上固定流程查好的证据：排名前 10 的病，每个都已经逐条核对过。

### 背景
固定流程（第 14 章）每道题都把前 10 名逐个核对。agent 却要自己一个个去查，查到第几个由它自己决定，于是有了 ④ 那 15 道：答案就在列表里，它没细查。

这部分固定流程已经做得很好，不用 agent 重复。直接把证据给它，它的 8 轮就能全部用在工具做不到的事上：想到排名漏掉的病，按名字查、再核对。

### 用了什么技术，为什么
- **第一条消息**：题目，加上固定流程的证据块。和第 14 章给模型看的一模一样，复用同一段代码
- **提示词**：开头和工具说明不变，后半段改成"证据已经在第一条消息里，把轮次花在工具做不到的事上"（版本 `seeded-v1`）
- **新系统叫 `seeded-agent`**，原来的 `agent` 保留，两个可以逐题比

### 公司里什么情况怎么做
- **固定流程和 agent 不是二选一**：能写死的部分写死（便宜、稳定），只把真正需要判断的部分交给 agent。Anthropic 和 OpenAI 给企业的建议都是这样
- **新旧两版并存**：名字不同、记录分开，才能逐题比

### 执行
从试跑记录里原样调出 agent 收到的第一条消息（只显示开头）。

In [ ]:
core = "rank_diseases, check_match, lookup_disease, find_disease"
seeded_ci = latest("seeded-agent", split="ci-20", tools=core)
trace = harness.traces(seeded_ci["run_id"])[0]
first_call = min((s for s in trace.data.spans if s.name == "Completions"), key=lambda s: s.start_time_ns)
message = first_call.inputs["messages"][1]["content"]
shown = message.splitlines()
n = sum(line.startswith("[") for line in shown)
print("\n".join(shown[:14]))
print(f"……（一共 {n} 个候选，这里只显示前两个）")

### 结果怎么看
- 开头是题目：性别、症状和它们的 HPO 编号（agent 调工具要用编号）
- 后面是候选：每个病解释了几个症状、哪些没解释、它常见而病人没有的症状
- agent 从这里出发，不用再自己排名、逐个核对

### 面试怎么说
> "The seeded agent starts where the fixed pipeline stops: its first message already holds the top ten candidates with their evidence, so its turns go to what the tools cannot do — the diseases they missed."

---

## 第 3 步 · 列不满 10 个就补满

### 在干什么
agent 列的答案不到 10 个时，按工具的顺序补满。

### 背景
第 15 章发现：200 道题里有 120 道，agent 只列了不到 10 个。前 10 名的位置空着，等于白白放弃。

### 用了什么技术，为什么
先列 agent 自己的答案，再按工具的顺序（先排名、再按解释了几个症状重排）往后补，跳过已经列过的。补进来的每一项，理由里写明"不是 agent 排的，是按工具的顺序补的"，看答案的人分得清哪些是 agent 的判断。

改之前先估算效果，不花钱：拿第 15 章 agent 的答案，在本机按同样的规则补满，再判分。

### 公司里什么情况怎么做
- **能用已有记录估算的，先估算**，不用重跑模型
- **规则补进来的要标清楚**：答案里哪些是模型的判断、哪些是规则补的，用户要分得清

### 执行
要在本机把 200 道题重新排名、核对一遍，大约一两分钟。

In [ ]:
import mlflow
from raredx.baselines.rank_then_check import RankThenCheck
from raredx.evals.harness import candidates_of
from raredx.evals.systems import question_of
from raredx.knowledge import masking
from raredx.tools.check_match import CheckMatch
from raredx.tools.rank_diseases import RankDiseases

seed = RankThenCheck(RankDiseases(hpo), CheckMatch(hpo))
mlflow.tracing.disable()          # 本机重算，不记追踪
short = before = after = 0
for trace in harness.traces(agent_run["run_id"]):
    truth = next(a.value for a in trace.info.assessments if a.name == "disease_id")
    case = by_id[trace.data.spans[0].inputs["question"]["case_id"]]
    listed = [c.disease_id for c in candidates_of(trace.data.spans[0].outputs)]
    filled = listed
    if len(listed) < 10:
        short += 1
        with masking.masked([case.pmid]):
            order = [m.disease_id for m, _ in seed.ordered(question_of(case))]
        filled = listed + [d for d in order if d not in listed]
    before += truth in listed[:10]
    after += truth in filled[:10]
mlflow.tracing.enable()
print(f"列了不到 10 个：{short} 道")
print(f"前 10 名：补之前 {before / 2:.1f}%，补满之后 {after / 2:.1f}%")

### 结果怎么看
- 前 10 名：54.0% → 56.0%。第 1 名不变：补的都排在 agent 自己的答案后面
- 这是用旧答案估算的。新开局的 agent 会给出不同的答案，真实效果第 20 章大考时再看

### 面试怎么说
> "The agent listed fewer than ten diagnoses in 120 of 200 cases. Filling the list from the tools' order, labelled as such, lifts top-10 from 54.0% to 56.0% on the recorded answers — measured by replay, at no cost."

---

## 第 4 步 · 小考：20 道题试跑

### 在干什么
用 ci-20 试跑新开局的 agent（约 ¥1）。

### 背景
每改一版，先在小题集上看能不能跑通：交了几道、有没有出错。20 道题太少，分数看不出高低。

### 执行

In [ ]:
rows = [("固定流程", latest("fixed-pipeline", split="ci-20")),
        ("agent（第 15 章）", latest("agent", split="ci-20", prompt_version="agent-v3")),
        ("新开局 agent", seeded_ci)]
for label, record in rows:
    m = harness.client.get_run(record["run_id"]).data.metrics
    print(f"{label:<12} 交了答案 {m['n_answered']:.0f}/20   第 1 名 {m['disease_top1/mean']:.0%}   前 10 名 {m['disease_top10/mean']:.0%}")

### 结果怎么看
- 新开局的 agent 20 道全部交了答案，没有出错：能跑通
- 分数先不看：20 道题的误差范围有正负 20 个百分点，差一两道题说明不了什么

### 面试怎么说
> "Every change gets a 20-case smoke run first — did it answer every case, did anything fail — before money goes into the 200-case evaluation."

---

## 第 5 步 · 定下工具清单和考法

### 在干什么
根据错题分析定下后面要做的工具，以及最后怎么考。

### 背景
一个很自然的想法是"能加的工具都加上"，公司里不这么做：
- 每个工具的说明，模型每一轮都要读一遍：多占地方，也多花钱
- 选项越多越容易走偏：我们的 agent 只有 8 轮，第 15 章的 v1 就是一直查、不交卷

所以每个工具都要对得上一类错题。

### 清单

| 章 | 工具 | 针对什么 |
|---|---|---|
| 17 | 找相似的病人：和一万个已发表的病人比症状 | ②（39 道，其中 33 道别的论文报过同病病人） |
| 18 | 疾病介绍（RAG）：在疾病的文字介绍里搜，回答带出处 | 给判断多一份证据；演示时每个诊断都有出处 |
| 19 | 基因工具：结合基因检测的结果 | ①（38 道） |

文献检索（PubMed）不做：答案常常就写在文献里，考试时遮不干净。

### 考法：建完再一起考，但每个工具值多少分要分得清
1. 每做完一个工具，用 ci-20 试跑一次（约 ¥1）：只看 agent 会不会用、有没有出错
2. 全部做完以后（第 20 章），用 dev-200 大考：带上全部工具考一次，再每次拿掉一个各考一次（消融实验）。拿掉一个工具分数掉多少，就是它值多少分；不掉分的就删掉
3. 定稿以后，用留出集正式考一次

### 公司里什么情况怎么做
- **消融实验（ablation）**：每次去掉一个部件看分数变化，是证明"每个部件都有用"的标准做法
- **工具少而精**：Anthropic 写工具的指南里专门提醒，常见的错误是把现有功能一股脑包成工具

### 面试怎么说
> "The tool list came from the error analysis, not a wish list: each new tool targets one class of misses, and the final evaluation ablates them one at a time, so every tool has to earn its place."

---

## 本章小结

| 做了什么 | 结果 |
|---|---|
| 错题分析 | 122 道错题：77 道错在"没找到"，45 道错在"没选对" |
| 新开局 | 第一条消息就带上固定流程的证据（`seeded-agent`） |
| 补满 10 个 | 用旧答案估算：前 10 名 54.0% → 56.0% |
| 小考 | 20 道全部交了答案，没有出错 |
| 工具清单 | 相似病人、疾病介绍（RAG）、基因工具；第 20 章一起大考，逐个拿掉看值多少分 |

## 面试题

1. **How did you decide what to improve next?**
   > "Error analysis, computed from the traces: for each miss, did the right disease ever appear in a tool output, and did the agent examine it? 77 of 122 misses never surfaced at all, so the next work went into retrieval, not prompting."

2. **Why not give the agent every tool you can think of?**
   > "Every tool's description is read on every turn, and more choices make a budget-limited agent wander — our first version never stopped investigating. Each new tool has to target a measured class of failures."

3. **How will you know each tool helps?**
   > "Ablation: the full system once, then once without each tool, on the same 200 cases, compared case by case. A tool whose removal costs nothing gets removed."

**下一章 17 · 找相似的病人**：② 类的 39 道题，正确答案在资料库里，却排得太靠后。很多这种病，别的论文报过病人：拿这个病人去和一万个已发表的病人比，看最像的那几个得的是什么病。